# 08. Supply Chain Security: Signing & SLSA

## Background

The 2020 SolarWinds attack demonstrated that compromising the *build process* rather than the application code itself can yield access to thousands of downstream targets. Supply chain security addresses the entire path from source commit to deployed artifact: every step where an attacker could inject malicious code or substitute a legitimate artifact.

**Sigstore** is the Linux Foundation's keyless signing infrastructure. Instead of managing long-lived signing keys, Sigstore ties signatures to ephemeral OIDC-issued certificates — your GitHub Actions workflow proves it is `github.com/your-org/your-repo` at the time of signing, and that proof is logged in Rekor (an append-only transparency log).

**SLSA** (Supply-chain Levels for Software Artifacts) is a security framework providing four levels of supply chain integrity guarantees, from basic provenance metadata (L1) to hermetic, isolated, reproducible builds (L4).

## What You'll Learn

- How artifact signing works: key pairs, signatures, and verification chains
- Sigstore's keyless signing model with OIDC identity binding
- SLSA levels and what build controls each level requires
- Verifying artifacts in a CI pipeline before deployment

## Why This Matters

ML artifacts (model weights, datasets, Docker images) are high-value targets. An attacker who substitutes a backdoored model file could compromise every system that downloads it. Signing and verification create an unforgeable chain of custody from training run to deployment.


## Artifact Signing (Simulated Sigstore Model)

In [ ]:
import hashlib
import hmac
import json
import time
import base64
import secrets
from dataclasses import dataclass, field
from typing import Optional, Dict

@dataclass
class SigningCertificate:
    subject: str        # OIDC identity, e.g. github-actions@repo
    issuer: str         # https://accounts.google.com or https://token.actions.githubusercontent.com
    not_before: float
    not_after: float    # Short-lived: not_before + 600 (10 min)
    public_key: str     # hex-encoded

@dataclass
class ArtifactSignature:
    artifact_digest: str   # SHA-256 of artifact bytes
    signature: str         # base64-encoded signature
    certificate: SigningCertificate
    rekor_log_index: int   # transparency log entry
    signed_at: float

class SimulatedKeylessSigner:
    '''Simulates Sigstore keyless signing flow:
    1. Get OIDC token from CI provider
    2. Exchange for short-lived signing certificate from Fulcio CA
    3. Sign artifact
    4. Submit to Rekor transparency log
    '''
    def __init__(self):
        self._master_secret = secrets.token_bytes(32)
        self._rekor_counter = 1000
        self._rekor_log: Dict[int, dict] = {}

    def _get_oidc_identity(self, workflow: str) -> str:
        return f'https://github.com/{workflow}/.github/workflows/release.yml@refs/heads/main'

    def _issue_certificate(self, identity: str) -> SigningCertificate:
        now = time.time()
        # Fulcio issues certs valid for only 10 minutes
        pub_key = hashlib.sha256((identity + str(now)).encode()).hexdigest()
        return SigningCertificate(
            subject=identity,
            issuer='https://token.actions.githubusercontent.com',
            not_before=now,
            not_after=now + 600,
            public_key=pub_key
        )

    def _sign(self, digest: str, cert: SigningCertificate) -> str:
        # Simulate ECDSA signature as HMAC for demonstration
        sig = hmac.new(self._master_secret,
                       f'{digest}:{cert.public_key}'.encode(),
                       hashlib.sha256).digest()
        return base64.b64encode(sig).decode()

    def _log_to_rekor(self, digest: str, sig: str, cert: SigningCertificate) -> int:
        idx = self._rekor_counter
        self._rekor_counter += 1
        self._rekor_log[idx] = {
            'index': idx,
            'artifact_digest': digest,
            'signature': sig[:16] + '...',
            'signer': cert.subject,
            'timestamp': time.time()
        }
        return idx

    def sign_artifact(self, artifact_bytes: bytes, workflow: str) -> ArtifactSignature:
        digest = hashlib.sha256(artifact_bytes).hexdigest()
        identity = self._get_oidc_identity(workflow)
        cert = self._issue_certificate(identity)
        sig = self._sign(digest, cert)
        log_idx = self._log_to_rekor(digest, sig, cert)

        return ArtifactSignature(
            artifact_digest=digest,
            signature=sig,
            certificate=cert,
            rekor_log_index=log_idx,
            signed_at=time.time()
        )

    def verify(self, artifact_bytes: bytes, sig_obj: ArtifactSignature) -> bool:
        # 1. Recompute digest
        digest = hashlib.sha256(artifact_bytes).hexdigest()
        if digest != sig_obj.artifact_digest:
            print('FAIL: digest mismatch (artifact tampered)')
            return False

        # 2. Check certificate not expired
        if time.time() > sig_obj.certificate.not_after:
            print('NOTE: certificate expired (normal for keyless — check Rekor instead)')

        # 3. Verify signature
        expected_sig = self._sign(digest, sig_obj.certificate)
        if not hmac.compare_digest(sig_obj.signature, expected_sig):
            print('FAIL: signature invalid')
            return False

        # 4. Check Rekor log inclusion
        if sig_obj.rekor_log_index not in self._rekor_log:
            print('FAIL: not found in transparency log')
            return False

        return True

signer = SimulatedKeylessSigner()

# Simulate signing a model artifact
model_bytes = b'model weights v1.0 (simulated binary)' * 100
sig = signer.sign_artifact(model_bytes, 'my-org/llm-training')

print(f'Artifact signed:')
print(f'  Digest:     sha256:{sig.artifact_digest[:16]}...')
print(f'  Signer:     {sig.certificate.subject}')
print(f'  Rekor idx:  {sig.rekor_log_index}')
print(f'  Cert TTL:   600s (keyless — identity bound, not key bound)')

# Verify
valid = signer.verify(model_bytes, sig)
print(f'\nVerification: {"PASS" if valid else "FAIL"}')

# Tamper with artifact
tampered = model_bytes + b'backdoor'
print('\nAttempting tampered artifact verification:')
valid_tampered = signer.verify(tampered, sig)
print(f'Tampered verification: {"PASS" if valid_tampered else "FAIL (expected)"}')


## SLSA Levels & Provenance

In [ ]:
from dataclasses import dataclass
from typing import List

@dataclass
class SLSAProvenance:
    '''SLSA provenance attestation (simplified SLSA v1.0 schema).'''
    build_type: str                # e.g. 'https://github.com/slsa-framework/slsa-github-generator'
    builder_id: str                # Immutable builder identity
    source_repo: str
    source_commit: str
    build_trigger: str             # push, pull_request, schedule
    build_invocation_id: str       # Unique per build run
    hermetic: bool                 # No network access during build
    isolated: bool                 # No other jobs in same environment
    reproducible: bool             # Same inputs -> same output bit-for-bit

SLSA_REQUIREMENTS = {
    'L1': {
        'description': 'Provenance exists',
        'requirements': ['provenance_generated', 'provenance_includes_expected_fields']
    },
    'L2': {
        'description': 'Hosted build, signed provenance',
        'requirements': ['provenance_generated', 'provenance_signed',
                         'hosted_build_service', 'provenance_non_falsifiable']
    },
    'L3': {
        'description': 'Hardened build, build-as-code',
        'requirements': ['provenance_generated', 'provenance_signed',
                         'hosted_build_service', 'provenance_non_falsifiable',
                         'build_instructions_in_repo', 'hermetic_build',
                         'ephemeral_isolated_environment']
    },
    'L4': {
        'description': 'Two-party review, hermetic reproducible build',
        'requirements': ['provenance_generated', 'provenance_signed',
                         'hosted_build_service', 'provenance_non_falsifiable',
                         'build_instructions_in_repo', 'hermetic_build',
                         'ephemeral_isolated_environment',
                         'two_person_reviewed', 'reproducible_build']
    }
}

def assess_slsa_level(provenance: SLSAProvenance, has_signing: bool,
                       build_instructions_in_repo: bool, two_person_review: bool) -> str:
    controls = {
        'provenance_generated': True,
        'provenance_includes_expected_fields': bool(provenance.source_repo and provenance.source_commit),
        'provenance_signed': has_signing,
        'hosted_build_service': 'github.com' in provenance.builder_id,
        'provenance_non_falsifiable': has_signing,
        'build_instructions_in_repo': build_instructions_in_repo,
        'hermetic_build': provenance.hermetic,
        'ephemeral_isolated_environment': provenance.isolated,
        'two_person_reviewed': two_person_review,
        'reproducible_build': provenance.reproducible,
    }

    achieved = 'L0'
    for level in ['L1', 'L2', 'L3', 'L4']:
        reqs = SLSA_REQUIREMENTS[level]['requirements']
        if all(controls.get(r, False) for r in reqs):
            achieved = level
        else:
            break

    print(f'SLSA Assessment: {achieved} — {SLSA_REQUIREMENTS.get(achieved, {}).get("description", "No provenance")}')
    print('\nControl status:')
    for ctrl, passed in controls.items():
        icon = 'PASS' if passed else 'FAIL'
        print(f'  [{icon}] {ctrl}')
    return achieved

prov = SLSAProvenance(
    build_type='https://github.com/slsa-framework/slsa-github-generator/generic@v1',
    builder_id='https://github.com/actions/runner@v2.304.0',
    source_repo='github.com/my-org/llm-training',
    source_commit='abc123def456',
    build_trigger='push',
    build_invocation_id='run-789012',
    hermetic=True,
    isolated=True,
    reproducible=False
)

level = assess_slsa_level(prov, has_signing=True, build_instructions_in_repo=True, two_person_review=False)
